In [1]:
%matplotlib inline
import os
import sys
from pathlib import Path
from urllib.parse import quote_plus

import pandas as pd


from dotenv import load_dotenv
from sqlalchemy import create_engine

_NOTEBOOK_DIR = Path.cwd()
_PROJECT_ROOT = _NOTEBOOK_DIR
for _ in range(5):
    if (_PROJECT_ROOT / ".env").exists():
        break
    _PROJECT_ROOT = _PROJECT_ROOT.parent

def _load_env():
    load_dotenv(_PROJECT_ROOT / ".env")

def _db_url():
    host = os.getenv("PGHOST")
    port = os.getenv("PGPORT")
    user = os.getenv("PGUSER")
    password = os.getenv("PGPASSWORD")
    dbname = os.getenv("PGDATABASE")
    pw = quote_plus(password) if password else ""
    return f"postgresql://{user}:{pw}@{host}:{port}/{dbname}"

_load_env()
engine = create_engine(_db_url())


In [2]:
from sklearn.metrics import log_loss, accuracy_score, mean_squared_error
from sklearn.model_selection import train_test_split

import numpy as np
import optuna
import xgboost as xgb

sys.path.insert(0, str(_NOTEBOOK_DIR))
from data_prep import load_pitcher_simulator_data, prepare_pitcher_features

print("imported modules")

imported modules


c:\Users\macia\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
df_raw = load_pitcher_simulator_data(engine)
X, y_dict = prepare_pitcher_features(df_raw)

print(f"Loaded {len(X)} pitches")
print(f"Features: {list(X.columns)}")
print(f"Targets: {list(y_dict.keys())}")

X_train, X_test, y_train_dict, y_test_dict = {}, {}, {}, {}
train_idx, test_idx = train_test_split(
    np.arange(len(X)),
    test_size=0.2,
    random_state=42,
    stratify=y_dict["pitch_type"],
)
X_train = X.iloc[train_idx].copy().reset_index(drop=True)
X_test = X.iloc[test_idx].copy().reset_index(drop=True)
for t in y_dict:
    y_train_dict[t] = y_dict[t].iloc[train_idx].reset_index(drop=True)
    y_test_dict[t] = y_dict[t].iloc[test_idx].reset_index(drop=True)

print(f"\nTrain: {len(X_train)} | Test: {len(X_test)}")

Loaded 493311 pitches
Features: ['pitcher', 'p_throws', 'stand', 'balls', 'strikes', 'is_pitcher_count', 'is_batter_count', 'inning', 'inning_topbot', 'outs_when_up', 'at_bat_number', 'pitch_number', 'previous_pitch_type', 'previous_release_speed', 'pitcher_pitches_this_game', 'pitcher_pitches_this_inning', 'game_date', 'home_team', 'away_team', 'game_type', 'pitcher_career_ip', 'pitcher_career_era', 'pitcher_career_so', 'pitcher_career_bb', 'pitcher_career_h', 'pitcher_career_er', 'pitcher_career_hr', 'pitcher_career_bfp', 'pitcher_career_ipouts']
Targets: ['pitch_type', 'plate_x', 'plate_z', 'release_speed', 'release_spin_rate']

Train: 394648 | Test: 98663


In [4]:
import json

repertoire_by_pitcher = (
    pd.DataFrame({"pitcher": X_train["pitcher"], "pitch_type": y_train_dict["pitch_type"]})
    .groupby("pitcher")["pitch_type"]
    .apply(lambda s: sorted(s.unique().tolist()))
    .to_dict()
)
# JSON keys must be strings
pitcher_repertoire_json = {str(int(k)): v for k, v in repertoire_by_pitcher.items()}
_SAVED = Path(_NOTEBOOK_DIR) / "saved_models"
_SAVED.mkdir(parents=True, exist_ok=True)
with open(_SAVED / "pitcher_repertoire.json", "w") as f:
    json.dump(pitcher_repertoire_json, f, indent=2)
print(f"Saved pitcher_repertoire.json ({len(pitcher_repertoire_json)} pitchers)")

Saved pitcher_repertoire.json (872 pitchers)


In [5]:
from sklearn.utils.class_weight import compute_class_weight

codes_in_train = sorted(y_train_dict["pitch_type"].unique())
type_to_idx = {t: i for i, t in enumerate(codes_in_train)}
map_pitch_types = {i: t for t, i in type_to_idx.items()}

y_train_pt = y_train_dict["pitch_type"].map(type_to_idx)
y_test_pt = y_test_dict["pitch_type"].map(lambda t: type_to_idx.get(t, 0))

classes_pt = np.unique(y_train_pt)
class_weights_pt = compute_class_weight(
    "balanced", classes=classes_pt, y=y_train_pt.to_numpy().ravel()
)
sample_weight_pt = class_weights_pt[np.searchsorted(classes_pt, y_train_pt.to_numpy().ravel())]

X_train_pt = X_train
y_train_pt_oversampled = y_train_pt

feats_s1 = list(X_train.columns)

In [ ]:
# Hyperparameter tuning: Optuna runs many trials (random sampling of n_estimators, max_depth,
# learning_rate) and we keep the config that minimizes log loss on the test set
def tune_pitch_type(trial):
    n_estimators = trial.suggest_int("n_estimators", 50, 500)
    max_depth = trial.suggest_int("max_depth", 3, 12)
    learning_rate = trial.suggest_float("learning_rate", 0.01, 0.3)
    subsample = trial.suggest_float("subsample", 0.5, 1.0)
    colsample_bytree = trial.suggest_float("colsample_bytree", 0.5, 1.0)
    model = xgb.XGBClassifier(
        objective="multi:softprob", random_state=42,
        n_estimators=n_estimators, max_depth=max_depth,
        learning_rate=learning_rate, subsample=subsample, colsample_bytree=colsample_bytree,
    )
    model.fit(X_train_pt[feats_s1], y_train_pt_oversampled)
    proba = model.predict_proba(X_test[feats_s1])
    # Pass labels so log_loss accepts when test set has fewer classes than train (e.g. 16 vs 17)
    return log_loss(y_test_pt, proba, labels=np.arange(len(codes_in_train)))

study_pt = optuna.create_study(direction="minimize")
study_pt.optimize(tune_pitch_type, n_trials=10)
print("Best log_loss:", study_pt.best_value, "| Best params:", study_pt.best_params)


[I 2026-02-23 02:47:03,075] A new study created in memory with name: no-name-c248b920-0ed9-4766-8ba7-d17c1c4fce20
[I 2026-02-23 02:47:47,121] Trial 0 finished with value: 1.2387862244924759 and parameters: {'n_estimators': 261, 'max_depth': 9, 'learning_rate': 0.16019314116602515, 'subsample': 0.7841638040794403, 'colsample_bytree': 0.7396763219851363}. Best is trial 0 with value: 1.2387862244924759.
[I 2026-02-23 02:48:19,081] Trial 1 finished with value: 1.3266124529794434 and parameters: {'n_estimators': 228, 'max_depth': 5, 'learning_rate': 0.2790672308485089, 'subsample': 0.6185584812964322, 'colsample_bytree': 0.9181385115047678}. Best is trial 0 with value: 1.2387862244924759.
[I 2026-02-23 02:48:37,900] Trial 2 finished with value: 1.369674435516077 and parameters: {'n_estimators': 151, 'max_depth': 4, 'learning_rate': 0.2723565020740682, 'subsample': 0.7020082690170416, 'colsample_bytree': 0.5785790464576148}. Best is trial 0 with value: 1.2387862244924759.
[I 2026-02-23 02:49

Best log_loss: 1.2387862244924759 | Best params: {'n_estimators': 261, 'max_depth': 9, 'learning_rate': 0.16019314116602515, 'subsample': 0.7841638040794403, 'colsample_bytree': 0.7396763219851363}


In [ ]:
# Addressing memorizing the majority / overfitting: we oversample rare pitch types (and cap
# common ones at the median count) so the model learns all types instead of memorizing noise
# or collapsing to the most frequent class
rs = np.random.RandomState(42)
n_classes = len(codes_in_train)
counts = y_train_pt.value_counts().reindex(range(n_classes), fill_value=0).values
target_per_class = int(np.median(counts[counts > 0]))  # avoid huge dataset
balanced_idx = []
for c in range(n_classes):
    idx_c = np.where(y_train_pt.values == c)[0]
    n_c = len(idx_c)
    if n_c == 0:
        continue
    if n_c >= target_per_class:
        chosen = rs.choice(idx_c, size=target_per_class, replace=False)
    else:
        chosen = rs.choice(idx_c, size=target_per_class, replace=True)
    balanced_idx.extend(chosen)
balanced_idx = np.array(balanced_idx)
rs.shuffle(balanced_idx)

X_train_pt = X_train.iloc[balanced_idx].reset_index(drop=True)
y_train_pt_oversampled = y_train_pt.iloc[balanced_idx].reset_index(drop=True)
print(f"Pitch-type training: original {len(X_train)} -> oversampled {len(X_train_pt)} (target {target_per_class} per class)")

Pitch-type training: original 394648 -> oversampled 113050 (target 6650 per class)


In [8]:
xgb_pt = xgb.XGBClassifier(objective="multi:softprob", random_state=42, **study_pt.best_params)
xgb_pt.fit(X_train_pt[feats_s1], y_train_pt_oversampled)

col_names = [map_pitch_types[i] for i in range(len(codes_in_train))]
proba_pt = xgb_pt.predict_proba(X_test[feats_s1])
pitch_type_pred = pd.Series(
    [col_names[i] for i in np.argmax(proba_pt, axis=1)],
    index=X_test.index,
)

_SAVED = Path(_NOTEBOOK_DIR) / "saved_models"
_SAVED.mkdir(parents=True, exist_ok=True)
xgb_pt.save_model(str(_SAVED / "pitcher_pitch_type.json"))
print("Saved pitcher_pitch_type.json")

Saved pitcher_pitch_type.json


In [9]:
X_train_s2 = X_train.copy()
X_train_s2["pitch_type_code"] = y_train_dict["pitch_type"].map(type_to_idx)

X_test_s2 = X_test.copy()
X_test_s2["pitch_type_code"] = pitch_type_pred.map(lambda t: type_to_idx.get(t, 0))

feats_s2 = feats_s1 + ["pitch_type_code"]

In [ ]:
# Hyperparameter tuning for the four regressors (plate_x, plate_z, release_speed, release_spin_rate)
# Optuna runs n_trials per target and we fit and save the model with the best params for each
def tune_reg(trial, target_name):
    n_estimators = trial.suggest_int("n_estimators", 50, 500)
    max_depth = trial.suggest_int("max_depth", 3, 12)
    learning_rate = trial.suggest_float("learning_rate", 0.01, 0.3)
    subsample = trial.suggest_float("subsample", 0.5, 1.0)
    colsample_bytree = trial.suggest_float("colsample_bytree", 0.5, 1.0)
    reg_alpha = trial.suggest_float("reg_alpha", 0.01, 10.0)
    reg_lambda = trial.suggest_float("reg_lambda", 0.1, 10.0)
    min_child_weight = trial.suggest_int("min_child_weight", 1, 10)
    model = xgb.XGBRegressor(objective="reg:squarederror", random_state=42,
        n_estimators=n_estimators, max_depth=max_depth,
        learning_rate=learning_rate, subsample=subsample, colsample_bytree=colsample_bytree,
        reg_alpha=reg_alpha, reg_lambda=reg_lambda, min_child_weight=min_child_weight)
    model.fit(X_train_s2[feats_s2], y_train_dict[target_name])
    pred = model.predict(X_test_s2[feats_s2])
    return np.sqrt(mean_squared_error(y_test_dict[target_name], pred))  # RMSE

reg_targets = ["plate_x", "plate_z", "release_speed", "release_spin_rate"]
reg_models = {}
for tgt in reg_targets:
    study = optuna.create_study(direction="minimize")
    study.optimize(lambda trial, t=tgt: tune_reg(trial, t), n_trials=10)
    model = xgb.XGBRegressor(objective="reg:squarederror", random_state=42, **study.best_params)
    model.fit(X_train_s2[feats_s2], y_train_dict[tgt])
    model.save_model(str(_SAVED / f"pitcher_{tgt}.json"))
    reg_models[tgt] = model
    print(f"pitcher_{tgt}.json: best RMSE = {study.best_value:.4f}")

[I 2026-02-23 02:57:39,137] A new study created in memory with name: no-name-a3417155-d79e-4086-beb2-111cd2504ccd
[I 2026-02-23 02:57:42,765] Trial 0 finished with value: 0.8403204820007729 and parameters: {'n_estimators': 265, 'max_depth': 10, 'learning_rate': 0.2509961513384511, 'subsample': 0.9490064723205153, 'colsample_bytree': 0.7157144620904249, 'reg_alpha': 5.196630307125014, 'reg_lambda': 5.443988902304089, 'min_child_weight': 9}. Best is trial 0 with value: 0.8403204820007729.
[I 2026-02-23 02:57:44,586] Trial 1 finished with value: 0.8171251161601277 and parameters: {'n_estimators': 217, 'max_depth': 7, 'learning_rate': 0.037584402131747664, 'subsample': 0.8549773915175687, 'colsample_bytree': 0.6003532626676915, 'reg_alpha': 8.957838376991306, 'reg_lambda': 5.267337494955374, 'min_child_weight': 6}. Best is trial 1 with value: 0.8171251161601277.
[I 2026-02-23 02:57:48,133] Trial 2 finished with value: 0.8209001898120147 and parameters: {'n_estimators': 163, 'max_depth': 11

pitcher_plate_x.json: best RMSE = 0.8160


[I 2026-02-23 02:58:09,725] Trial 0 finished with value: 1.0351342708104079 and parameters: {'n_estimators': 466, 'max_depth': 4, 'learning_rate': 0.2699709371984859, 'subsample': 0.9688126399422601, 'colsample_bytree': 0.9551779755134692, 'reg_alpha': 7.119989858533233, 'reg_lambda': 0.11458053744688709, 'min_child_weight': 2}. Best is trial 0 with value: 1.0351342708104079.
[I 2026-02-23 02:58:19,918] Trial 1 finished with value: 1.059473657738564 and parameters: {'n_estimators': 440, 'max_depth': 12, 'learning_rate': 0.12220922306472208, 'subsample': 0.8840419330231908, 'colsample_bytree': 0.9201442686536037, 'reg_alpha': 4.524913858737833, 'reg_lambda': 8.262579841913716, 'min_child_weight': 9}. Best is trial 0 with value: 1.0351342708104079.
[I 2026-02-23 02:58:22,531] Trial 2 finished with value: 1.0342057306782992 and parameters: {'n_estimators': 120, 'max_depth': 12, 'learning_rate': 0.07098472186615015, 'subsample': 0.9511528973764274, 'colsample_bytree': 0.6112383252107823, '

pitcher_plate_z.json: best RMSE = 0.9781


[I 2026-02-23 02:58:47,002] Trial 0 finished with value: 7.348984774591524 and parameters: {'n_estimators': 391, 'max_depth': 11, 'learning_rate': 0.025884046527594073, 'subsample': 0.6336573039314906, 'colsample_bytree': 0.8155044497219814, 'reg_alpha': 5.42163918111552, 'reg_lambda': 6.030209455612658, 'min_child_weight': 8}. Best is trial 0 with value: 7.348984774591524.
[I 2026-02-23 02:58:47,805] Trial 1 finished with value: 7.221866988099468 and parameters: {'n_estimators': 105, 'max_depth': 4, 'learning_rate': 0.2157931510418956, 'subsample': 0.5047307237076211, 'colsample_bytree': 0.5012496772880598, 'reg_alpha': 1.8855001796094797, 'reg_lambda': 3.2740835405922715, 'min_child_weight': 8}. Best is trial 1 with value: 7.221866988099468.
[I 2026-02-23 02:58:51,062] Trial 2 finished with value: 7.323242823207238 and parameters: {'n_estimators': 483, 'max_depth': 7, 'learning_rate': 0.09378803723017805, 'subsample': 0.954063101346406, 'colsample_bytree': 0.5072820773035028, 'reg_al

pitcher_release_speed.json: best RMSE = 6.9988


[I 2026-02-23 02:59:09,837] Trial 0 finished with value: 416.5402812812975 and parameters: {'n_estimators': 252, 'max_depth': 6, 'learning_rate': 0.1476208364787648, 'subsample': 0.6589811623627817, 'colsample_bytree': 0.5322281389938445, 'reg_alpha': 2.8582534858033677, 'reg_lambda': 1.100165510534301, 'min_child_weight': 9}. Best is trial 0 with value: 416.5402812812975.
[I 2026-02-23 02:59:12,384] Trial 1 finished with value: 416.7771054405355 and parameters: {'n_estimators': 457, 'max_depth': 4, 'learning_rate': 0.22048525850305778, 'subsample': 0.8329065765226635, 'colsample_bytree': 0.956665393251318, 'reg_alpha': 1.5410947522979548, 'reg_lambda': 4.533130449383998, 'min_child_weight': 6}. Best is trial 0 with value: 416.5402812812975.
[I 2026-02-23 02:59:13,833] Trial 2 finished with value: 414.3492769638 and parameters: {'n_estimators': 209, 'max_depth': 5, 'learning_rate': 0.12978799007408873, 'subsample': 0.580489598504299, 'colsample_bytree': 0.6008427646952899, 'reg_alpha':

pitcher_release_spin_rate.json: best RMSE = 414.3493


In [11]:
#compute how well the pitch type model and the four regressors do on the test set
acc_pt = accuracy_score(y_test_pt, xgb_pt.predict(X_test[feats_s1]))
proba_pt_test = xgb_pt.predict_proba(X_test[feats_s1])
y_test_onehot = np.zeros_like(proba_pt_test)
for i, pt in enumerate(y_test_pt):
    if 0 <= pt < proba_pt_test.shape[1]:
        y_test_onehot[i, int(pt)] = 1
loss_pt = log_loss(y_test_onehot, proba_pt_test)
print("Pitch type: accuracy =", f"{acc_pt:.4f}", "| log loss =", f"{loss_pt:.4f}")

for tgt in reg_targets:
    pred = reg_models[tgt].predict(X_test_s2[feats_s2])
    rmse = np.sqrt(mean_squared_error(y_test_dict[tgt], pred))
    mae = np.abs(y_test_dict[tgt].values - pred).mean()
    print(f"{tgt}: RMSE = {rmse:.4f} | MAE = {mae:.4f}")

Pitch type: accuracy = 0.3599 | log loss = 1.7307
plate_x: RMSE = 0.8160 | MAE = 0.6454
plate_z: RMSE = 0.9781 | MAE = 0.7689
release_speed: RMSE = 6.9988 | MAE = 5.5144
release_spin_rate: RMSE = 414.3493 | MAE = 286.0668


In [ ]:
# Monte Carlo at inference + avoiding memorizing noise: compute the std of prediction errors
# per pitch type and save it. The app then adds random noise with this scale when predicting,
# so the simulator outputs a distribution of plausible outcomes instead of a single deterministic
# prediction that would overfit to training noise
import json

pt_train = y_train_dict["pitch_type"]  # pitch type strings
residual_stds = {}
DEFAULT_STDS = {"plate_x": 0.5, "plate_z": 0.5, "release_speed": 1.5, "release_spin_rate": 200}
MIN_SAMPLES = 10

for tgt in reg_targets:
    pred = reg_models[tgt].predict(X_train_s2[feats_s2])
    res = y_train_dict[tgt].values - pred
    df_res = pd.DataFrame({"pt": pt_train.values, "res": res})
    by_pt = df_res.groupby("pt")["res"].agg(["std", "count"])
    global_std = df_res["res"].std()
    std_by_pt = {}
    for pt in by_pt.index:
        row = by_pt.loc[pt]
        std_by_pt[pt] = float(row["std"]) if row["count"] >= MIN_SAMPLES and pd.notna(row["std"]) else float(global_std)
    residual_stds[tgt] = std_by_pt

with open(_SAVED / "residual_stds.json", "w") as f:
    json.dump(residual_stds, f, indent=2)
print("Saved residual_stds.json")

Saved residual_stds.json


In [13]:
#for each pitcher and each pitch type compute average location and save so the app can nudge predictions toward it
MIN_PITCHES_FOR_MEANS = 20
df_loc = pd.DataFrame({
    "pitcher": X_train["pitcher"],
    "pitch_type": y_train_dict["pitch_type"],
    "plate_x": y_train_dict["plate_x"],
    "plate_z": y_train_dict["plate_z"],
})
by_pitcher_pt = df_loc.groupby(["pitcher", "pitch_type"]).agg(
    plate_x=("plate_x", "mean"),
    plate_z=("plate_z", "mean"),
    n=("plate_x", "count"),
).reset_index()
pitcher_plate_means = {}
for _, row in by_pitcher_pt.iterrows():
    if row["n"] < MIN_PITCHES_FOR_MEANS:
        continue
    pid = str(int(row["pitcher"]))
    pt = row["pitch_type"]
    if pid not in pitcher_plate_means:
        pitcher_plate_means[pid] = {}
    pitcher_plate_means[pid][pt] = {"plate_x": float(row["plate_x"]), "plate_z": float(row["plate_z"])}
with open(_SAVED / "pitcher_plate_means.json", "w") as f:
    json.dump(pitcher_plate_means, f, indent=2)
print(f"Saved pitcher_plate_means.json ({len(pitcher_plate_means)} pitchers with >= {MIN_PITCHES_FOR_MEANS} pitches per type)")

Saved pitcher_plate_means.json (758 pitchers with >= 20 pitches per type)


In [14]:
#list the model and json files we wrote to saved_models
list(_SAVED.glob("pitcher_*.json"))

[WindowsPath('c:/Users/macia/Documents/Final Year project/Final_Year_Project/Models/Training/saved_models/pitcher_pitch_type.json'),
 WindowsPath('c:/Users/macia/Documents/Final Year project/Final_Year_Project/Models/Training/saved_models/pitcher_plate_means.json'),
 WindowsPath('c:/Users/macia/Documents/Final Year project/Final_Year_Project/Models/Training/saved_models/pitcher_plate_x.json'),
 WindowsPath('c:/Users/macia/Documents/Final Year project/Final_Year_Project/Models/Training/saved_models/pitcher_plate_z.json'),
 WindowsPath('c:/Users/macia/Documents/Final Year project/Final_Year_Project/Models/Training/saved_models/pitcher_release_speed.json'),
 WindowsPath('c:/Users/macia/Documents/Final Year project/Final_Year_Project/Models/Training/saved_models/pitcher_release_spin_rate.json'),
 WindowsPath('c:/Users/macia/Documents/Final Year project/Final_Year_Project/Models/Training/saved_models/pitcher_repertoire.json')]

In [15]:
#for each pitcher compute how often they threw each pitch type and save so the app can blend with model probs
import json
df_pt = pd.DataFrame({"pitcher": X_train["pitcher"], "pitch_type": y_train_dict["pitch_type"]})
rates = df_pt.groupby("pitcher")["pitch_type"].value_counts(normalize=True).unstack(fill_value=0.0)
pitcher_pitch_type_rates = {}
for pid in rates.index:
    d = rates.loc[pid].to_dict()
    pitcher_pitch_type_rates[str(int(pid))] = {str(k): float(v) for k, v in d.items()}
with open(_SAVED / "pitcher_pitch_type_rates.json", "w") as f:
    json.dump(pitcher_pitch_type_rates, f, indent=2)
print(f"Saved pitcher_pitch_type_rates.json ({len(pitcher_pitch_type_rates)} pitchers)")

Saved pitcher_pitch_type_rates.json (872 pitchers)
